# Per-fold F-max scoring

Computes weighted F-max for each base model × each CV fold × each GO aspect
(MFO / BPO / CCO), using the precomputed outputs from
`src/create_ensemble_datasets.py`:

- `train_merged.tsv` — one confidence column per base model
- `train_terms.tsv` — ground-truth GO annotations
- `IA.txt` — information-accretion weights
- `splits/f{k}_split_{mf,bp,cc}.csv` — per-aspect fold assignments

For aspect _A_ at fold _k_, we use that aspect's fold file to pick the
validation proteins (this is the pool of proteins actually annotated in _A_),
filter both the per-model predictions and the ground truth to those proteins,
then call `fmax(...)` and extract `scores[A]`.

Result is written to `data/final_data/fmax_per_fold.json` with shape
`{model: {aspect: [fmax_fold0, ..., fmax_foldK-1]}}`.


In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

from src.metrics import fmax, ASPECTS

In [2]:
# ---- config ----
DATA_DIR   = Path('data/final_data')
SPLITS_DIR = DATA_DIR / 'splits'
N_FOLDS    = 5
OUTPUT_JSON = DATA_DIR / 'fmax_per_fold.json'

# aspect code in train_terms.tsv -> suffix used in split filenames
ASPECT_SUFFIX = {'MFO': 'mf', 'BPO': 'bp', 'CCO': 'cc'}

In [3]:
# ---- load precomputed data ----
train_merged = pd.read_csv(DATA_DIR / 'train_merged.tsv', sep='\t')
print(f"train_merged: {len(train_merged):>10,} rows, "
      f"{train_merged['protein_id'].nunique():,} proteins")

train_terms = pd.read_csv(DATA_DIR / 'train_terms.tsv', sep='\t')
print(f"train_terms:  {len(train_terms):>10,} annotations, "
      f"{train_terms['EntryID'].nunique():,} proteins")

# IA.txt is tab-separated: term\tia
ia_weights = {}
with open(DATA_DIR / 'IA.txt') as f:
    for line in f:
        parts = line.strip().split('\t')
        if len(parts) == 2:
            ia_weights[parts[0]] = float(parts[1])
print(f"ia_weights:   {len(ia_weights):>10,} terms")

# model names come from the conf_<name> columns
model_cols  = [c for c in train_merged.columns if c.startswith('conf_')]
model_names = [c[len('conf_'):] for c in model_cols]
print(f"models:       {model_names}")

train_merged: 123,405,629 rows, 142,246 proteins
train_terms:   5,363,863 annotations, 142,246 proteins
ia_weights:       43,248 terms
models:       ['sequence', 'structure', 'protgoat']


In [4]:
# ---- pre-index ground truth by aspect (small speedup over repeated filtering) ----
gt_by_aspect = {a: train_terms[train_terms['aspect'] == a] for a in ASPECTS}
for a, df in gt_by_aspect.items():
    print(f"  {a}: {len(df):>10,} annotations, {df['EntryID'].nunique():,} proteins")

  MFO:    670,114 annotations, 78,637 proteins
  BPO:  3,497,732 annotations, 92,210 proteins
  CCO:  1,196,017 annotations, 92,912 proteins


In [5]:
# ---- compute F-max for each (model, fold, aspect) ----
# results[model][aspect] -> list of length N_FOLDS
results = {m: {a: [] for a in ASPECTS} for m in model_names}

for aspect in ASPECTS:
    suffix    = ASPECT_SUFFIX[aspect]
    aspect_gt = gt_by_aspect[aspect]
    print(f"\n=== {aspect} (suffix=_{suffix}) ===")

    for fold in range(N_FOLDS):
        split_path = SPLITS_DIR / f'f{fold}_split_{suffix}.csv'
        if not split_path.exists():
            print(f"  fold {fold}: {split_path} missing — skipping")
            continue

        split        = pd.read_csv(split_path)
        val_proteins = set(split.loc[split['split'] == 'val', 'protein_id'])

        # Restrict predictions and ground truth to this fold's validation proteins
        val_preds = train_merged[train_merged['protein_id'].isin(val_proteins)]
        val_gt    = aspect_gt[aspect_gt['EntryID'].isin(val_proteins)]
        print(f"  fold {fold}: {len(val_proteins):,} val proteins, "
              f"{len(val_gt):,} GT annotations, {len(val_preds):,} pred rows")

        for model in model_names:
            # one model's predictions, drop zeros for speed (threshold sweep starts at 0.01)
            preds = (
                val_preds[['protein_id', 'GO_term', f'conf_{model}']]
                .rename(columns={f'conf_{model}': 'confidence'})
            )
            preds = preds[preds['confidence'] > 0]

            scores = fmax(preds, val_gt, ia_weights)
            f_asp  = float(scores[aspect])
            results[model][aspect].append(f_asp)
            print(f"      {model:<28} fmax = {f_asp:.4f}")


=== MFO (suffix=_mf) ===
  fold 0: 15,728 val proteins, 134,626 GT annotations, 14,286,181 pred rows


/Users/shanewilliams/GradSchool/Spring2026/CompBio/final_project/comp-bio-167/src/metrics.py:40: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: dict(zip(g['GO_term'], g['confidence'])))


      sequence                     fmax = 0.2377


/Users/shanewilliams/GradSchool/Spring2026/CompBio/final_project/comp-bio-167/src/metrics.py:40: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: dict(zip(g['GO_term'], g['confidence'])))


      structure                    fmax = 0.1688


...

      sequence                     fmax = 0.1878


/Users/shanewilliams/GradSchool/Spring2026/CompBio/final_project/comp-bio-167/src/metrics.py:40: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: dict(zip(g['GO_term'], g['confidence'])))


      structure                    fmax = 0.1673


/Users/shanewilliams/GradSchool/Spring2026/CompBio/final_project/comp-bio-167/src/metrics.py:40: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: dict(zip(g['GO_term'], g['confidence'])))


      protgoat                     fmax = 0.4269


In [6]:
# ---- save ----
OUTPUT_JSON.parent.mkdir(parents=True, exist_ok=True)
with open(OUTPUT_JSON, 'w') as f:
    json.dump(results, f, indent=2)
print(f"Saved -> {OUTPUT_JSON}")

Saved -> data/final_data/fmax_per_fold.json


In [7]:
# ---- summary: mean ± std across folds, per model per aspect ----
header = f"{'model':<28}" + "".join(f"{a:>18}" for a in ASPECTS) + f"{'overall':>12}"
print(header)
print('-' * len(header))
for model in model_names:
    per_aspect_means = []
    row = f"{model:<28}"
    for aspect in ASPECTS:
        vals = np.array(results[model][aspect])
        row += f"{vals.mean():.4f}±{vals.std():.4f}".rjust(18)
        per_aspect_means.append(vals.mean())
    row += f"{np.mean(per_aspect_means):.4f}".rjust(12)
    print(row)

model                                      MFO               BPO               CCO     overall
----------------------------------------------------------------------------------------------
sequence                         0.2385±0.0015     0.1624±0.0008     0.1887±0.0012      0.1966
structure                        0.1680±0.0021     0.1769±0.0007     0.1658±0.0015      0.1702
protgoat                         0.4107±0.0017     0.4805±0.0013     0.4283±0.0020      0.4399
